# RAPTOR Tree Explorer

Browse the hierarchical RAPTOR tree structure stored in Chroma database.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from collections import Counter

# Connect to the backend Chroma database
client = chromadb.PersistentClient(path='backend/chromadb')
print('✓ Available collections:', client.list_collections())

# Load the RAPTOR collection
col = client.get_collection('finfrag_raptor')
print(f'✓ Collection: {col.name}')
print(f'✓ Total documents: {col.count()}')

## Root Nodes (Level 1 Summaries)

In [ ]:
# Fetch root nodes (parent_id = '__raptor_root__')
root_result = col.get(
    where={'parent_id': '__raptor_root__'},
    include=['documents', 'metadatas'],
    limit=20
)

print(f'Root nodes: {len(root_result["ids"])}\n')
for i, (doc_id, doc, meta) in enumerate(zip(root_result['ids'], root_result['documents'], root_result['metadatas']), 1):
    print(f'{i}. {doc_id}')
    print(f'   Level: {meta.get("level")}, Parent: {meta.get("parent_id")}')
    print(f'   {doc[:270].replace(chr(10), " ")}...\n')

## Children of First Root Node

In [ ]:
# Inspect children of the first root node
if root_result['ids']:
    first_root_id = root_result['ids'][0]
    print(f'Parent: {first_root_id}\n')
    
    children = col.get(
        where={'parent_id': first_root_id},
        include=['documents', 'metadatas'],
        limit=20
    )
    
    print(f'Children: {len(children["ids"])}\n')
    for i, (doc_id, doc, meta) in enumerate(zip(children['ids'], children['documents'], children['metadatas']), 1):
        print(f'{i}. {doc_id}')
        print(f'   Level: {meta.get("level")}')
        print(f'   {doc[:240].replace(chr(10), " ")}...\n')

## Tree Statistics

In [ ]:
all_docs = col.get(include=['metadatas'], limit=5000)
metadatas = all_docs['metadatas']

levels = Counter(m.get('level') for m in metadatas)
summary_types = Counter(m.get('is_summary', False) for m in metadatas)

print('RAPTOR Tree Statistics')
print(f'Total nodes: {len(all_docs["ids"])}')
print(f'Levels: {dict(sorted(levels.items()))}')
print(f'Summaries: {summary_types[True]}')
print(f'Leaf nodes: {summary_types[False]}')